In [ ]:
# ==================================
# 1. Import libraries
# ==================================
import pandas as pd
import warnings
import joblib
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE


# ==================================
# 2. Load dataset
# ==================================
df = pd.read_csv("Customer-Churn-Records.csv")

print("Dataset Shape:", df.shape)
print("Columns:", df.columns)


# ==================================
# 3. Drop unnecessary columns
# ==================================
#df.drop(["RowNumber","CustomerId","Surname"], axis=1, inplace=True, errors="ignore")
# Drop leakage columns
df.drop(["Complain", "Satisfaction Score", "Point Earned"], axis=1, inplace=True, errors="ignore")
df.drop(["Surname", "Card Type"], axis=1, inplace=True, errors="ignore")
# ==================================
# 4. Handle missing values
# ==================================
df.fillna(method='ffill', inplace=True)


# ==================================
# 5. Encode categorical variables
# ==================================
# Convert ALL categorical columns automatically
df = pd.get_dummies(df, drop_first=True)


# ==================================
# 6. Feature Engineering
# ==================================
if "Balance" in df.columns and "EstimatedSalary" in df.columns:
    df["BalanceSalaryRatio"] = df["Balance"]/(df["EstimatedSalary"]+1)

if "Tenure" in df.columns and "Age" in df.columns:
    df["TenureAgeRatio"] = df["Tenure"]/(df["Age"]+1)


# ==================================
# 7. Define features & target
# ==================================
if "Exited" not in df.columns:
    raise ValueError("Target column 'Exited' missing!")

X = df.drop("Exited", axis=1)
y = df["Exited"]

# Ensure numeric (fix SMOTE error)
X = X.apply(pd.to_numeric, errors='coerce')
X.fillna(0, inplace=True)

print("Feature shape:", X.shape)
print("Target distribution:\n", y.value_counts())


# ==================================
# 8. Train-test split
# ==================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Split done!")


# ==================================
# 9. Apply SMOTE (ONLY on train)
# ==================================
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

print("After SMOTE:", X_train.shape)


# ==================================
# 10. Feature scaling
# ==================================
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# ==================================
# 11. Train models (avoid overfitting)
# ==================================
rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=8,
    random_state=42
)

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    eval_metric="logloss"
)

rf.fit(X_train, y_train)
xgb.fit(X_train, y_train)


# ==================================
# 12. Predictions
# ==================================
rf_train_pred = rf.predict(X_train)
rf_test_pred = rf.predict(X_test)

xgb_train_pred = xgb.predict(X_train)
xgb_test_pred = xgb.predict(X_test)


# ==================================
# 13. Evaluation
# ==================================
print("\n------ Random Forest ------")
print("Train Accuracy:", accuracy_score(y_train, rf_train_pred))
print("Test Accuracy:", accuracy_score(y_test, rf_test_pred))
print(classification_report(y_test, rf_test_pred))
print("ROC AUC:", roc_auc_score(y_test, rf_test_pred))


print("\n------ XGBoost ------")
print("Train Accuracy:", accuracy_score(y_train, xgb_train_pred))
print("Test Accuracy:", accuracy_score(y_test, xgb_test_pred))
print(classification_report(y_test, xgb_test_pred))
print("ROC AUC:", roc_auc_score(y_test, xgb_test_pred))

# Save model
joblib.dump(rf, "rf_model.pkl")
joblib.dump(xgb, "bank_churn_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(X.columns.tolist(), "features.pkl")

print("Model saved successfully")

Dataset Shape: (10000, 18)
Columns: Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited', 'Complain',
       'Satisfaction Score', 'Card Type', 'Point Earned'],
      dtype='object')
Feature shape: (10000, 15)
Target distribution:
 Exited
0    7962
1    2038
Name: count, dtype: int64
Split done!
After SMOTE: (12710, 15)

------ Random Forest ------
Train Accuracy: 0.8653815892997639
Test Accuracy: 0.821
              precision    recall  f1-score   support

           0       0.91      0.86      0.89      1607
           1       0.54      0.65      0.59       393

    accuracy                           0.82      2000
   macro avg       0.72      0.76      0.74      2000
weighted avg       0.84      0.82      0.83      2000

ROC AUC: 0.7559769519801252

------ XGBoost ------
Train Accuracy: 0.8852871754523997
Test Accuracy: 0.829
           

In [ ]:
import streamlit as st
import numpy as np
import joblib

# Load models
rf_model = joblib.load("rf_model.pkl")
xgb_model = joblib.load("xgb_model.pkl")
scaler = joblib.load("scaler.pkl")

st.set_page_config(page_title="Customer Churn Prediction", layout="centered")

st.title("🏦 Bank Customer Churn Prediction")
st.write("Predict whether a customer will churn or not")

# -----------------------------
# User Inputs
# -----------------------------
credit_score = st.number_input("Credit Score", 300, 900, 600)
age = st.slider("Age", 18, 100, 35)
tenure = st.slider("Tenure (years)", 0, 10, 3)
balance = st.number_input("Balance", 0.0, 300000.0, 50000.0)
num_products = st.slider("Number of Products", 1, 4, 1)
has_card = st.selectbox("Has Credit Card", [0, 1])
is_active = st.selectbox("Is Active Member", [0, 1])
salary = st.number_input("Estimated Salary", 0.0, 200000.0, 50000.0)

# Geography encoding
geography = st.selectbox("Geography", ["France", "Germany", "Spain"])
geo_germany = 1 if geography == "Germany" else 0
geo_spain = 1 if geography == "Spain" else 0

# Gender encoding
gender = st.selectbox("Gender", ["Male", "Female"])
gender_male = 1 if gender == "Male" else 0

# Feature Engineering
balance_salary_ratio = balance / (salary + 1)
tenure_age_ratio = tenure / (age + 1)

# Final input array (must match training features)
features = np.array([[credit_score, age, tenure, balance, num_products,
                      has_card, is_active, salary,
                      geo_germany, geo_spain,
                      gender_male,
                      balance_salary_ratio, tenure_age_ratio]])

# Scale
features = scaler.transform(features)

# -----------------------------
# Prediction
# -----------------------------
model_choice = st.radio("Choose Model", ["Random Forest", "XGBoost"])

if st.button("Predict"):
    if model_choice == "Random Forest":
        prediction = rf_model.predict(features)[0]
        prob = rf_model.predict_proba(features)[0][1]
    else:
        prediction = xgb_model.predict(features)[0]
        prob = xgb_model.predict_proba(features)[0][1]

    if prediction == 1:
        st.error(f"⚠️ Customer will CHURN (Probability: {prob:.2f})")
    else:
        st.success(f"✅ Customer will NOT churn (Probability: {prob:.2f})")